{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# YouTube -> Shorts Pipeline (Kaggle Runner)\n",
    "\n",
    "Run cells top to bottom. Each session:\n",
    "1. Clone/pull latest code from GitHub\n",
    "2. Install dependencies\n",
    "3. Stage 1: download + transcribe\n",
    "4. Stage 2: LLM clip selection\n",
    "5. Stage 3: reframe + caption + export\n",
    "6. Cleanup raw files\n",
    "7. Download your finished clips as a zip\n",
    "\n",
    "**Before running:** enable GPU under Settings (right panel) -> Accelerator -> GPU T4 x2."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ---- CONFIG ----\n",
    "GITHUB_REPO_URL = \"https://github.com/kunj-web/yt-shorts-pipeline.git\"\n",
    "REPO_DIR = \"/kaggle/working/yt-shorts-pipeline\"\n",
    "\n",
    "RAW_DIR = \"/kaggle/working/raw\"\n",
    "TRANSCRIPTS_DIR = \"/kaggle/working/transcripts\"\n",
    "CLIP_PICKS_DIR = \"/kaggle/working/clip_picks\"\n",
    "CLIPS_OUTPUT_DIR = \"/kaggle/working/clips\"\n",
    "TMP_RENDER_DIR = \"/kaggle/working/tmp_render\"\n",
    "MODELS_DIR = \"/kaggle/working/models\"\n",
    "\n",
    "LLAMA_MODEL_REPO = \"bartowski/Meta-Llama-3.1-8B-Instruct-GGUF\"\n",
    "LLAMA_MODEL_FILE = \"Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf\""
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ---- 1. Get latest code ----\n",
    "import os\n",
    "\n",
    "if not os.path.exists(REPO_DIR):\n",
    "    !git clone {GITHUB_REPO_URL} {REPO_DIR}\n",
    "else:\n",
    "    !cd {REPO_DIR} && git pull"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ---- 2. Install dependencies ----\n",
    "!pip install -q yt-dlp faster-whisper opencv-python mediapipe llama-cpp-python huggingface_hub"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ---- 3. Confirm GPU is available ----\n",
    "!nvidia-smi"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Stage 1: Download + Transcribe\n",
    "\n",
    "Edit `config/videos.txt` in the repo (or push updated URLs via git) before running this."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "VIDEOS_TXT = f\"{REPO_DIR}/config/videos.txt\"\n",
    "\n",
    "!python {REPO_DIR}/stage1_ingest/download_videos.py \\\n",
    "    --input {VIDEOS_TXT} \\\n",
    "    --output {RAW_DIR} \\\n",
    "    --max-height 1080"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!python {REPO_DIR}/stage1_ingest/transcribe_audio.py \\\n",
    "    --input-dir {RAW_DIR} \\\n",
    "    --output-dir {TRANSCRIPTS_DIR} \\\n",
    "    --model-size medium \\\n",
    "    --device cuda \\\n",
    "    --compute-type float16"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Stage 2: LLM Clip Selection"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# One-time-per-session download of the quantized Llama model (~4.9GB)\n",
    "import os\n",
    "\n",
    "os.makedirs(MODELS_DIR, exist_ok=True)\n",
    "model_path = f\"{MODELS_DIR}/{LLAMA_MODEL_FILE}\"\n",
    "\n",
    "if not os.path.exists(model_path):\n",
    "    !huggingface-cli download {LLAMA_MODEL_REPO} {LLAMA_MODEL_FILE} --local-dir {MODELS_DIR}\n",
    "else:\n",
    "    print(\"Model already present this session.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!python {REPO_DIR}/stage2_select/select_viral_clips.py \\\n",
    "    --transcripts-dir {TRANSCRIPTS_DIR} \\\n",
    "    --output-dir {CLIP_PICKS_DIR} \\\n",
    "    --model-path {MODELS_DIR}/{LLAMA_MODEL_FILE} \\\n",
    "    --min-seconds 30 \\\n",
    "    --max-seconds 90 \\\n",
    "    --clips-per-video 3"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Stage 3: Reframe + Caption + Export"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!python {REPO_DIR}/stage3_render/export_clips.py \\\n",
    "    --raw-dir {RAW_DIR} \\\n",
    "    --transcripts-dir {TRANSCRIPTS_DIR} \\\n",
    "    --clip-picks-dir {CLIP_PICKS_DIR} \\\n",
    "    --output-dir {CLIPS_OUTPUT_DIR} \\\n",
    "    --tmp-dir {TMP_RENDER_DIR}"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cleanup: delete raw source videos once clips are exported"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!python {REPO_DIR}/utils/cleanup_raw_files.py \\\n",
    "    --raw-dir {RAW_DIR} \\\n",
    "    --output-dir {CLIPS_OUTPUT_DIR} \\\n",
    "    --tmp-dir {TMP_RENDER_DIR}"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Zip and download your finished clips\n",
    "\n",
    "Run this last -- it packages everything in `clips/` into a single zip you can download from Kaggle's Output panel (right sidebar)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import shutil\n",
    "\n",
    "zip_path = \"/kaggle/working/clips_export\"\n",
    "shutil.make_archive(zip_path, \"zip\", CLIPS_OUTPUT_DIR)\n",
    "print(f\"Zipped clips -> {zip_path}.zip\")\n",
    "print(\"Download it from the Output panel on the right side of the Kaggle notebook page.\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}